# Agent as Tool 패턴 - 완전한 멀티에이전트 시스템



In [ ]:
!pip install --upgrade boto3 botocore strands-agents -q
print('⚠️ Kernel > Restart Kernel 실행 후 다음 셀부터 실행')

In [ ]:
import boto3, json, time, uuid, hashlib
from strands import Agent, tool
from strands.models import BedrockModel
from strands.handlers.callback_handler import PrintingCallbackHandler
from strands.agent.conversation_manager import SlidingWindowConversationManager
from typing import Dict, Any
from botocore.config import Config

REGION = 'us-east-1'
S3_VECTORS_BUCKET = 'twl-marengo-3'
S3_VECTORS_INDEX = 'marengo3'
DYNAMODB_TABLE = 'strands-video-tasks'
MARENGO_MODEL_ID = 'twelvelabs.marengo-embed-3-0-v1:0'
PEGASUS_MODEL_ID = 'us.twelvelabs.pegasus-1-2-v1:0'
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

bedrock_runtime = boto3.client('bedrock-runtime', region_name=REGION, config=Config(signature_version='v4'))
s3 = boto3.client('s3', region_name=REGION)
s3vectors = boto3.client('s3vectors', region_name=REGION)
dynamodb = boto3.resource('dynamodb', region_name=REGION)
task_table = dynamodb.Table(DYNAMODB_TABLE)
transcribe = boto3.client('transcribe', region_name=REGION)
s3_sigv4 = boto3.client('s3', region_name=REGION, config=Config(signature_version='s3v4'))

claude_model = BedrockModel(model_id='us.anthropic.claude-sonnet-4-20250514-v1:0', region_name=REGION)
print('✅ 초기화 완료')

## 1단계: 모든 기본 도구들 정의

In [ ]:
# Embedding 도구
@tool
def create_video_embedding(s3_uri: str) -> Dict[str, Any]:
    '''비디오 임베딩 생성 (완료까지 대기 후 S3 Vectors 저장).'''
    task_id = str(uuid.uuid4())[:8]
    bucket = s3_uri.split('/')[2]
    key = '/'.join(s3_uri.split('/')[3:])
    task_table.put_item(Item={'task_id': task_id, 's3_uri': s3_uri, 's3_bucket': bucket, 's3_key': key, 'status': 'processing', 'created_at': int(time.time())})
    
    response = bedrock_runtime.start_async_invoke(
        modelId=MARENGO_MODEL_ID,
        modelInput={'inputType': 'video', 'video': {'mediaSource': {'s3Location': {'uri': s3_uri, 'bucketOwner': ACCOUNT_ID}}, 'embeddingOption': ['visual', 'audio'], 'embeddingScope': ['clip'], 'segmentation': {'method': 'fixed', 'fixed': {'durationSec': 6}}}},
        outputDataConfig={'s3OutputDataConfig': {'s3Uri': f's3://{bucket}/embeddings/{task_id}/'}}
    )
    invocation_arn = response['invocationArn']
    task_table.update_item(Key={'task_id': task_id}, UpdateExpression='SET invocation_arn = :arn', ExpressionAttributeValues={':arn': invocation_arn})
    
    print(f'⏳ 임베딩 처리 중... (task_id: {task_id})')
    for _ in range(60):
        status = bedrock_runtime.get_async_invoke(invocationArn=invocation_arn)['status']
        if status == 'Completed': break
        if status in ['Failed', 'Expired']: return {'error': f'임베딩 실패: {status}', 'task_id': task_id}
        time.sleep(10)
    else:
        return {'error': '타임아웃', 'task_id': task_id}
    
    output_uri = bedrock_runtime.get_async_invoke(invocationArn=invocation_arn)['outputDataConfig']['s3OutputDataConfig']['s3Uri']
    prefix = '/'.join(output_uri.split('/')[3:])
    objs = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    json_key = next((o['Key'] for o in objs.get('Contents', []) if o['Key'].endswith('output.json')), None)
    data = json.loads(s3.get_object(Bucket=bucket, Key=json_key)['Body'].read())
    clips = data.get('data', [])
    
    vectors = [{'key': f"{task_id}_{c['embeddingOption']}_{c['startSec']}_{c['endSec']}", 'data': {'float32': c['embedding']}, 'metadata': {'task_id': task_id, 'embeddingOption': c['embeddingOption'], 'startSec': c['startSec'], 'endSec': c['endSec']}} for c in clips]
    for i in range(0, len(vectors), 100): s3vectors.put_vectors(vectorBucketName=S3_VECTORS_BUCKET, indexName=S3_VECTORS_INDEX, vectors=vectors[i:i+100])
    
    task_table.update_item(Key={'task_id': task_id}, UpdateExpression='SET #s = :s, clip_count = :c', ExpressionAttributeNames={'#s': 'status'}, ExpressionAttributeValues={':s': 'completed', ':c': len(clips)})
    return {'task_id': task_id, 'status': 'completed', 's3_uri': s3_uri, 'stored_clips': len(clips)}

print('✅ Embedding 도구 정의 완료')

In [ ]:
# Search 도구들 (10개까지 검색 가능)
@tool
def search_video_clips(query: str, top_k: int = 50, max_results: int = 10) -> Dict[str, Any]:
    '''텍스트로 비디오 클립 검색. 중복 제거 후 최대 10개 결과 반환.'''
    response = bedrock_runtime.invoke_model(modelId=MARENGO_MODEL_ID, body=json.dumps({'inputType': 'text', 'text': {'inputText': query}}), contentType='application/json')
    emb = json.loads(response['body'].read())['data'][0]['embedding']
    
    results = s3vectors.query_vectors(vectorBucketName=S3_VECTORS_BUCKET, indexName=S3_VECTORS_INDEX, queryVector={'float32': emb}, topK=top_k, returnDistance=True, returnMetadata=True, filter={'embeddingOption': {'$in': ['visual', 'audio']}})
    
    clips = []
    seen_intervals = []
    
    for v in results.get('vectors', []):
        if len(clips) >= max_results:  # 동적 제한
            break
            
        meta = v.get('metadata', {})
        task_info = task_table.get_item(Key={'task_id': meta.get('task_id')}).get('Item', {})
        start, end = int(meta.get('startSec', 0)), int(meta.get('endSec', 0))
        
        # 중복 체크: 3초 이내 겹치는 구간 제거
        is_duplicate = any(abs(start - s) <= 3 or abs(end - e) <= 3 for s, e in seen_intervals)
        
        if not is_duplicate:
            seen_intervals.append((start, end))
            bucket, key = task_info.get('s3_bucket', ''), task_info.get('s3_key', '')
            clips.append({
                'video': key, 
                's3_bucket': bucket, 
                'type': meta.get('embeddingOption'), 
                'timestamp': f'{start//60}:{start%60:02d}-{end//60}:{end%60:02d}', 
                'start_sec': start, 
                'end_sec': end,
                'similarity_score': v.get('distance', 0)
            })
    
    return {'query': query, 'clips': clips}

@tool
def get_clip_playback_url(s3_bucket: str, s3_key: str, start_sec: int, end_sec: int) -> Dict[str, Any]:
    '''S3 영상의 특정 구간 재생 가능한 presigned URL 생성.'''
    base_url = s3_sigv4.generate_presigned_url('get_object', Params={'Bucket': s3_bucket, 'Key': s3_key}, ExpiresIn=3600)
    return {'playback_url': f'{base_url}#t={start_sec},{end_sec}'}

print('✅ Search 도구들 정의 완료')

In [ ]:
# Summary 도구
@tool
def summarize_video(s3_uri: str, prompt: str = '이 영상을 챕터를 구분해서 3문장 정도로 요약해줘') -> Dict[str, Any]:
    '''Pegasus 모델로 영상 요약 생성.'''
    response = bedrock_runtime.invoke_model(
        modelId=PEGASUS_MODEL_ID,
        body=json.dumps({
            'inputPrompt': prompt,
            'mediaSource': {'s3Location': {'uri': s3_uri, 'bucketOwner': ACCOUNT_ID}}
        }),
        contentType='application/json'
    )
    result = json.loads(response['body'].read())
    return {'s3_uri': s3_uri, 'summary': result.get('message', result.get('response', result))}

print('✅ Summary 도구 정의 완료')

In [ ]:
# Transcript 도구들
def _get_transcript_data(s3_uri: str):
    bucket = s3_uri.split('/')[2]
    job_name = f"transcript-{hashlib.md5(s3_uri.encode()).hexdigest()[:8]}"
    return json.loads(s3.get_object(Bucket=bucket, Key=f'transcripts/{job_name}.json')['Body'].read())

def _ensure_transcript(s3_uri: str) -> Dict[str, Any]:
    '''자막 파일 확인/생성 (내부용).'''
    bucket = s3_uri.split('/')[2]
    job_name = f"transcript-{hashlib.md5(s3_uri.encode()).hexdigest()[:8]}"
    output_key = f'transcripts/{job_name}.json'
    try:
        status = transcribe.get_transcription_job(TranscriptionJobName=job_name)['TranscriptionJob']['TranscriptionJobStatus']
    except transcribe.exceptions.BadRequestException:
        key = '/'.join(s3_uri.split('/')[3:])
        ext = key.split('.')[-1].lower()
        media_format = 'mp4' if ext in ['mov', 'mp4', 'm4a'] else ext
        transcribe.start_transcription_job(TranscriptionJobName=job_name, Media={'MediaFileUri': s3_uri}, MediaFormat=media_format, LanguageCode='ko-KR', OutputBucketName=bucket, OutputKey=output_key)
        status = 'IN_PROGRESS'
    if status == 'IN_PROGRESS':
        print(f'⏳ 자막 생성 중... (job: {job_name})')
        while status == 'IN_PROGRESS':
            time.sleep(5)
            status = transcribe.get_transcription_job(TranscriptionJobName=job_name)['TranscriptionJob']['TranscriptionJobStatus']
    return {'transcript_file': f's3://{bucket}/{output_key}'} if status == 'COMPLETED' else {'error': '자막 생성 실패'}

@tool
def get_transcript_file(s3_uri: str) -> Dict[str, Any]:
    '''자막 파일 S3 경로 반환 (없으면 생성).'''
    return _ensure_transcript(s3_uri)

@tool
def get_transcript(s3_uri: str) -> Dict[str, Any]:
    '''자막을 타임스탬프와 함께 테이블 형식으로 반환. 없으면 자동 생성.'''
    try:
        data = _get_transcript_data(s3_uri)
    except:
        result = _ensure_transcript(s3_uri)
        if 'error' in result:
            return result
        data = _get_transcript_data(s3_uri)
    grouped = {}
    for item in data['results']['items']:
        if item['type'] == 'pronunciation':
            sec = int(float(item.get('start_time', 0)))
            key = f"{sec//5*5//60}:{sec//5*5%60:02d}"
            grouped[key] = grouped.get(key, '') + ' ' + item['alternatives'][0]['content']
    return {'transcript': [{'시간': k, '자막': v.strip()} for k, v in grouped.items()]}

@tool
def get_keywords(s3_uri: str) -> Dict[str, Any]:
    '''영상 자막에서 키워드 추출. 없으면 자동 생성.'''
    try:
        data = _get_transcript_data(s3_uri)
    except:
        result = _ensure_transcript(s3_uri)
        if 'error' in result:
            return result
        data = _get_transcript_data(s3_uri)
    transcript = ' '.join([i['alternatives'][0]['content'] for i in data['results']['items'] if i['type'] == 'pronunciation'])[:2000]
    response = bedrock_runtime.invoke_model(modelId='us.anthropic.claude-3-5-haiku-20241022-v1:0',
        body=json.dumps({'anthropic_version': 'bedrock-2023-05-31', 'max_tokens': 256,
            'messages': [{'role': 'user', 'content': f'핵심 키워드 10개 JSON 배열로만:\n{transcript}'}]}),
        contentType='application/json')
    return {'keywords': json.loads(response['body'].read())['content'][0]['text']}

print('✅ Transcript 도구들 정의 완료')

## 2단계: 전문 에이전트들 생성

In [ ]:
# Video Analysis Agent (임베딩 + 요약)
video_analysis_agent = Agent(
    model=claude_model,
    tools=[create_video_embedding, summarize_video],
    system_prompt='당신은 영상 분석 전문 에이전트입니다. 영상 임베딩 생성과 요약을 담당합니다.'
)

# Search Agent (검색 + 재생 URL)
search_agent = Agent(
    model=claude_model,
    tools=[search_video_clips, get_clip_playback_url],
    system_prompt='당신은 영상 검색 전문 에이전트입니다. 영상 클립 검색과 재생 URL 생성을 담당합니다. 검색 결과를 찾으면 자동으로 재생 URL도 생성해주세요.'
)

# Transcript Agent (자막 + 키워드)
transcript_agent = Agent(
    model=claude_model,
    tools=[get_transcript_file, get_transcript, get_keywords],
    system_prompt='당신은 자막 처리 전문 에이전트입니다. 자막 생성, 조회, 키워드 추출을 담당합니다.'
)

print('✅ 전문 에이전트들 생성 완료')

## 3단계: Agent as Tool 래핑

In [ ]:
@tool
def video_analysis_tool(query: str) -> str:
    """
    영상 분석 도구 (임베딩 생성, 요약)
    
    Use this tool for video embedding creation and summarization.
    
    Args:
        query: The analysis request (String)
    
    Returns:
        Analysis results (String)
    """
    print("🎬 Video Analysis Agent 호출")
    response = video_analysis_agent(query)
    print(f"✅ Video Analysis Agent 응답 완료")
    return str(response)

@tool
def video_search_tool(query: str) -> str:
    """
    영상 검색 도구 (클립 검색, 재생 URL 생성)
    
    Use this tool for searching video clips and generating playback URLs.
    
    Args:
        query: The search request (String)
    
    Returns:
        Search results with playback links (String)
    """
    print("🔍 Video Search Agent 호출")
    response = search_agent(query)
    print(f"✅ Video Search Agent 응답 완료")
    return str(response)

@tool
def transcript_tool(query: str) -> str:
    """
    자막 처리 도구 (자막 생성/조회, 키워드 추출)
    
    Use this tool for transcript generation, retrieval, and keyword extraction.
    
    Args:
        query: The transcript request (String)
    
    Returns:
        Transcript or keyword results (String)
    """
    print("📝 Transcript Agent 호출")
    response = transcript_agent(query)
    print(f"✅ Transcript Agent 응답 완료")
    return str(response)

print('✅ Agent as Tool 래핑 완료')

## 4단계: 오케스트레이터 생성

In [ ]:
ORCHESTRATOR_SYSTEM_PROMPT = """
You are the Video Processing Orchestrator Agent. Always respond in Korean.

Your key responsibilities:
1. Use video_analysis_tool for video embedding creation and summarization
2. Use video_search_tool for searching video clips and generating playback URLs
3. Use transcript_tool for transcript generation, retrieval, and keyword extraction
4. Coordinate between different agents to provide comprehensive video analysis

Tool Selection Guidelines:
- For embedding creation or video summarization → video_analysis_tool
- For searching clips or goal scenes → video_search_tool
- For transcripts, subtitles, or keywords → transcript_tool
- For complex requests, use multiple tools in sequence

Always provide clear, helpful responses to users.
"""

orchestrator = Agent(
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
    tools=[video_analysis_tool, video_search_tool, transcript_tool],
    model=claude_model,
    callback_handler=PrintingCallbackHandler(),
    conversation_manager=SlidingWindowConversationManager(window_size=20, should_truncate_results=True)
)

print('✅ Agent as Tool 패턴 완전 구현 완료')

## 5단계: 테스트 실행

In [ ]:
# 테스트 1: 골 장면 검색 (10개까지 가능)
#response = orchestrator("골 넣는 장면 10개 찾아줘")
#print(response)

In [ ]:
# 테스트 2: 복합 요청 (요약 + 키워드 + 검색)
response = orchestrator("{S3 URI} 내용 요약해주고 키워드 찾아줘, 그 후에 골대에 공이 정확히 들어가는 장면 찾아줘")
print(response)